In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
TRAIN_DIR = "/content/drive/MyDrive/disaster_final/disaster_final/train"
VAL_DIR = "/content/drive/MyDrive/disaster_final/disaster_final/validation"
TEST_DIR = "/content/drive/MyDrive/disaster_final/disaster_final/test"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 40

PROJECTION_DIM = 128
TEMPERATURE = 0.07
LR = 1e-3

In [ ]:
DISASTER_TYPES = ["earthquake","flood","wildfire"]
SEVERITY_LEVELS = ["low","medium","high"]

COMBINED_LABEL = {}

idx = 0
for d in DISASTER_TYPES:
    for s in SEVERITY_LEVELS:
        COMBINED_LABEL[(d,s)] = idx
        idx += 1

print(COMBINED_LABEL)

{('earthquake', 'low'): 0, ('earthquake', 'medium'): 1, ('earthquake', 'high'): 2, ('flood', 'low'): 3, ('flood', 'medium'): 4, ('flood', 'high'): 5, ('wildfire', 'low'): 6, ('wildfire', 'medium'): 7, ('wildfire', 'high'): 8}


In [ ]:
def load_paths(dataset_dir):

    paths=[]
    labels=[]

    for disaster in DISASTER_TYPES:
        for severity in SEVERITY_LEVELS:

            folder=os.path.join(dataset_dir,disaster,severity)

            if not os.path.exists(folder):
                continue

            for f in os.listdir(folder):

                if f.lower().endswith((".jpg",".jpeg",".png")):

                    paths.append(os.path.join(folder,f))
                    labels.append(COMBINED_LABEL[(disaster,severity)])

    return np.array(paths),np.array(labels)


train_paths,train_labels = load_paths(TRAIN_DIR)
val_paths,val_labels = load_paths(VAL_DIR)
test_paths,test_labels = load_paths(TEST_DIR)

print("Train:",len(train_paths))
print("Validation:",len(val_paths))
print("Test:",len(test_paths))

Train: 2244
Validation: 892
Test: 462


In [ ]:
def load_image(path):

    img = tf.io.read_file(path)

    # Decode image properly
    img = tf.image.decode_jpeg(img, channels=3)

    # Set shape explicitly
    img.set_shape([None, None, 3])

    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    img = tf.cast(img, tf.float32) / 255.0

    return img

In [ ]:
def augment(img):

    img=tf.image.random_flip_left_right(img)

    img=tf.image.random_brightness(img,0.3)
    img=tf.image.random_contrast(img,0.7,1.3)

    return img

In [ ]:
def make_views(path,label):

    img=load_image(path)

    v1=augment(img)
    v2=augment(img)

    return (v1,v2),label

In [ ]:
train_ds=tf.data.Dataset.from_tensor_slices((train_paths,train_labels))

train_ds=train_ds.shuffle(2000)

train_ds=train_ds.map(
    make_views,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds=train_ds.batch(BATCH_SIZE)
train_ds=train_ds.prefetch(tf.data.AUTOTUNE)


val_ds=tf.data.Dataset.from_tensor_slices((val_paths,val_labels))

val_ds=val_ds.map(
    make_views,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds=val_ds.batch(BATCH_SIZE)

In [ ]:
inputs=tf.keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))

base_model=EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE,IMG_SIZE,3)
)

base_model.trainable=True

x=base_model(inputs)

x=GlobalAveragePooling2D()(x)

x=Dense(512,activation="relu")(x)

x=Dense(PROJECTION_DIM)(x)

z=Lambda(lambda t: tf.math.l2_normalize(t,axis=1))(x)

contrastive_model=Model(inputs,z)

contrastive_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,771,107 (18.20 MB)

 Trainable params: 4,729,084 (18.04 MB)

 Non-trainable params: 42,023 (164.16 KB)

In [ ]:
class SupConLoss(tf.keras.losses.Loss):

    def call(self,labels,embeddings):

        labels=tf.concat([labels,labels],0)

        sim=tf.matmul(
            embeddings,
            embeddings,
            transpose_b=True
        )

        sim=sim/TEMPERATURE

        mask=tf.cast(
            tf.equal(
                tf.expand_dims(labels,1),
                tf.expand_dims(labels,0)
            ),
            tf.float32
        )

        logits_mask=1-tf.eye(tf.shape(labels)[0])

        mask=mask*logits_mask

        exp=tf.exp(sim)*logits_mask

        log_prob=sim-tf.math.log(
            tf.reduce_sum(exp,axis=1,keepdims=True)+1e-8
        )

        mean_log=tf.reduce_sum(mask*log_prob,axis=1)/(tf.reduce_sum(mask,axis=1)+1e-8)

        return -tf.reduce_mean(mean_log)

In [ ]:
optimizer=Adam(LR)
loss_fn=SupConLoss()

for epoch in range(EPOCHS):

    losses=[]

    for (v1,v2),label in train_ds:

        with tf.GradientTape() as tape:

            z1=contrastive_model(v1,training=True)
            z2=contrastive_model(v2,training=True)

            embeddings=tf.concat([z1,z2],axis=0)

            loss=loss_fn(label,embeddings)

        grads=tape.gradient(
            loss,
            contrastive_model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(grads,contrastive_model.trainable_variables)
        )

        losses.append(loss.numpy())

    print("Epoch",epoch+1,"Loss:",np.mean(losses))

Epoch 1 Loss: 3.4526615
Epoch 2 Loss: 2.9742944
Epoch 3 Loss: 2.766695
Epoch 4 Loss: 2.6289175
Epoch 5 Loss: 2.398809
Epoch 6 Loss: 2.2783885
Epoch 7 Loss: 2.1971323
Epoch 8 Loss: 2.1556067
Epoch 9 Loss: 2.1475935
Epoch 10 Loss: 2.105459
Epoch 11 Loss: 2.151508
Epoch 12 Loss: 2.097436
Epoch 13 Loss: 2.0760698
Epoch 14 Loss: 2.10333
Epoch 15 Loss: 2.04421
Epoch 16 Loss: 1.9885608
Epoch 17 Loss: 2.0081077
Epoch 18 Loss: 2.0745838
Epoch 19 Loss: 2.127657
Epoch 20 Loss: 2.0520601
Epoch 21 Loss: 2.033856
Epoch 22 Loss: 2.0932677
Epoch 23 Loss: 2.0526373
Epoch 24 Loss: 2.0237548
Epoch 25 Loss: 2.072297
Epoch 26 Loss: 2.0354078
Epoch 27 Loss: 2.0610304
Epoch 28 Loss: 2.0091836
Epoch 29 Loss: 1.9771048
Epoch 30 Loss: 1.998201
Epoch 31 Loss: 1.9630264
Epoch 32 Loss: 1.97407
Epoch 33 Loss: 2.0113337
Epoch 34 Loss: 1.9936665
Epoch 35 Loss: 2.0851839
Epoch 36 Loss: 2.081855
Epoch 37 Loss: 2.0370584
Epoch 38 Loss: 2.0236213
Epoch 39 Loss: 2.0315788
Epoch 40 Loss: 1.989333


In [ ]:
gap_output = contrastive_model.layers[-4].output

backbone_model = Model(
    inputs=contrastive_model.input,
    outputs=gap_output
)

backbone_model.save("/content/Backbone_model_V-Zeus.keras")

print("Backbone model version-Zeus saved!")

Backbone model version-Zeus saved!
